# two universal potentials against each other in Li$_{10}$GeP$_2$S$_{12}$ (no DFT)

For each model in `JOBS`: `B` replicas of the 200-atom cell (2×2×1 of mp-696128) at 800 K, Langevin dynamics with weak friction (1 ps$^{-1}$), every atom stored every 20 fs together with the model's own energy. Then every model evaluates the energy of every other model's frames. Last, 16 independent 50 ps runs of MACE-MPA-0, each analysed like one ab initio run, give the scatter of a single 50 ps diffusion coefficient.

Input: `lgps_bundle.zip` (`batched_md.py`, `foundation_md.py`, `lgps_md.py`, the LGPS CIF). Runtime: GPU. Output in the Drive folder `lgps_md`: `traj_<model>.npz`, `cross_<A>_on_<B>.npy`, `noise_<model>.npz`. Finished pieces are skipped and an interrupted trajectory resumes from its checkpoint when the cells are run again.


In [ ]:
!pip -q install mace-torch chgnet
import torch, time, os, numpy as np
print(torch.__version__, torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU')

In [ ]:
JOBS = ['MACE-MPA-0-medium', 'MACE-OMAT-0-medium', 'CHGNet-MPtrj']
B, PS = 8, 100                 # replicas per model, ps per replica after 10 ps of equilibration
NOISE_MODEL, NOISE_B, NOISE_PS = 'MACE-MPA-0-medium', 16, 50
OUT = '.'
try:
    from google.colab import drive; drive.mount('/content/drive'); OUT = '/content/drive/MyDrive/lgps_md'; os.makedirs(OUT, exist_ok=True)
except Exception as err: print('no Drive, working locally:', err)
if not os.path.exists('lgps_md.py'):
    from google.colab import files
    up = files.upload()            # choose lgps_bundle.zip
    os.system('unzip -o -q lgps_bundle.zip')
print(OUT, sorted(os.listdir('.')))

In [ ]:
import lgps_md as g
DEV = 'cuda' if torch.cuda.is_available() else 'cpu'
atoms = g.lgps_cell(); print(len(atoms), atoms.get_chemical_formula(), atoms.cell.lengths())
# speed and time estimate
for name in JOBS:
    s = g.speed(name, atoms, B, DEV); n = (PS + 10) * 1e3 / g.DT_FS
    print(f'{name:20s} {s * 1e3:7.1f} ms per step for {B} replicas  ->  {s * n / 3600:.1f} h for {PS} ps')
s = g.speed(NOISE_MODEL, atoms, NOISE_B, DEV); print(f'noise runs: {s * (NOISE_PS + 5) * 1e3 / g.DT_FS / 3600:.1f} h')

In [ ]:
for name in JOBS:
    g.run_model(name, atoms, OUT, B=B, ps=PS, dev=DEV)

In [ ]:
g.noise_runs(NOISE_MODEL, atoms, OUT, B=NOISE_B, ps=NOISE_PS, dev=DEV)

In [ ]:
for a in JOBS:
    for b in JOBS:
        if a != b: g.cross_energies(a, b, OUT, B=B, dev=DEV)
print(sorted(os.listdir(OUT)))